# Exploration des Donnees — Enquete Serologique COVID-19
### Analyse descriptive complete de la population etudiee

---

> **Contexte :** Ce notebook presente l'exploration complete d'un jeu de donnees
> issu d'une enquete de seroprevalence COVID-19 en milieu communautaire.

| Section | Contenu |
|---------|----------|
| **1** | Chargement & Apercu general |
| **2** | Description des variables |
| **3** | Profil socio-demographique |
| **4** | Distribution geographique |
| **5** | Analyse serologique |
| **6** | Symptomes |
| **7** | Comorbidites |
| **8** | Comportements preventifs |
| **9** | Analyses croisees (serologie x facteurs) |
| **10** | Profil des cas positifs |
| **11** | Synthese & Conclusions |


---
## 0. Importation des Bibliotheques & Configuration


In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd          # Manipulation de donnees
import numpy as np           # Calculs numeriques
from scipy import stats      # Tests statistiques

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px                   # Graphiques interactifs
import plotly.graph_objects as go             # Graphiques personnalises
from plotly.subplots import make_subplots     # Sous-graphiques

# -- Configuration graphique globale ----
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130})

# -- Palette de couleurs ----------------
SERO_COLORS = {'Positif': '#E74C3C', 'Negatif': '#2ECC71'}
age_order   = ['Enfant', 'Adolescent', 'Adulte', 'Personne agee']



---
## 1. Chargement des Donnees & Apercu General

> Ce qu'on fait ici :
> - Chargement du fichier Excel
> - Verification des dimensions, types et valeurs manquantes
> - Premier apercu statistique


In [2]:
# -- Chargement -----------------
# Le fichier doit etre dans le meme dossier que ce notebook
# 1. Chargement des données
df = pd.read_excel("C:/Users/achao/Desktop/STAGE/Covid_Niger/Files of pj/base_covid_population_vrai/Base_population_Netoyée Pop_VF.xlsx")
n, c = df.shape
print(f'Dataset charge : {n} individus x {c} variables')


Dataset charge : 4318 individus x 36 variables


In [3]:
# -- Apercu des premieres lignes ------
display(df.head())


,Code,Longitude,Latitude,Quartier_corrige,Population,Sexe,Age,Categorie_age,Profession,Statut_matrimonial,...,Frissons,Nausées,Dyspnée,Vomissements,Sueurs,Eruption cutanée,Odynophagie,Conjonctivite,Rhinorragie,Serologie
0,14-01,2.09289,13.48281,Banga bana,24700,Femme,19,Adolescent,Elève/Etudiant,Marié,...,Non,Non,Non,Non,Non,Non,Non,Non,Non,Positif
1,15-02,2.09289,13.48281,Banga bana,24700,Femme,60,Personne âgée,Sans emploi,Veuf,...,Non,Non,Non,Non,Non,Non,Non,Non,Non,Positif
2,15-03,2.09289,13.48281,Banga bana,24700,Homme,6,Enfant,Sans emploi,Célibataire,...,Non,Non,Non,Non,Non,Non,Non,Non,Non,Positif
3,15-05,2.09289,13.48281,Banga bana,24700,Femme,16,Adolescent,Sans emploi,Marié,...,Non,Non,Non,Non,Non,Oui,Non,Non,Non,Positif
4,15-06,2.09289,13.48281,Banga bana,24700,Homme,2,Enfant,Sans emploi,Célibataire,...,Non,Non,Non,Non,Non,Non,Non,Non,Non,Négatif


In [4]:
# -- Types de variables ---------------
# object = texte/categoriel | int64/float64 = numerique
print('Types de variables :')
display(df.dtypes.to_frame(name='Type').T)


Types de variables :


,Code,Longitude,Latitude,Quartier_corrige,Population,Sexe,Age,Categorie_age,Profession,Statut_matrimonial,...,Frissons,Nausées,Dyspnée,Vomissements,Sueurs,Eruption cutanée,Odynophagie,Conjonctivite,Rhinorragie,Serologie
Type,object,float64,float64,object,int64,object,int64,object,object,object,...,object,object,object,object,object,object,object,object,object,object


In [5]:
# -- Valeurs manquantes ---------------
# Etape indispensable avant toute analyse
missing     = df.isnull().sum()
missing_pct = (missing / n * 100).round(2)
miss_df     = pd.DataFrame({'Valeurs manquantes': missing,
                            'Pourcentage (%)': missing_pct})
miss_df = miss_df[miss_df['Valeurs manquantes'] > 0]

if miss_df.empty:
    print('Aucune valeur manquante dans le dataset -- donnees completes.')
else:
    print('Variables avec valeurs manquantes :')
    display(miss_df)


Aucune valeur manquante dans le dataset -- donnees completes.


In [6]:
# -- Statistiques descriptives --------
print('Statistiques des variables numeriques :')
display(df.describe().round(2))


Statistiques des variables numeriques :


,Longitude,Latitude,Population,Age
count,4318.00,4318.00,4318.00,4318.00
mean,2.15,13.52,18758.28,26.58
std,1.49,0.20,11705.70,21.38
min,1.10,2.10,1559.00,1.00
25%,2.09,13.49,11497.00,9.00
50%,2.11,13.53,17085.00,18.00
75%,2.14,13.55,28428.00,42.00
max,92.13,16.51,48551.00,100.00


---
## 2. Description & Categorisation des Variables

Le jeu de donnees contient **5 categories de variables** :

| # | Categorie | Variables | Role |
|---|-----------|-----------|------|
| 1 | **Geographiques** | Code, Longitude, Latitude, Quartier_corrige, Population | Localisation spatiale |
| 2 | **Socio-demographiques** | Sexe, Age, Categorie_age, Profession, Statut_matrimonial | Structure de la population |
| 3 | **Comorbidites** | Diabte, HTA, Maladie_cardiaque | Antecedents medicaux |
| 4 | **Symptomes** | Cephalee, Fievre, Toux, Anosmie... (18 vars) | Tableau clinique |
| 5 | **Comportements preventifs** | Port masque, Gel, Savon... | Mesures barrieres |
| **Cible** | **Serologie** | Positif / Negatif | Resultat du test |


In [7]:
df.columns

Index(['Code', 'Longitude', 'Latitude', 'Quartier_corrige', 'Population',
       'Sexe', 'Age', 'Categorie_age', 'Profession', 'Statut_matrimonial',
       'Diabte', 'HTA', 'Maladie_cardiaque', 'Port du masque à domicile',
       'Port masque exterieur', 'Usage du Gel hydroalcoolique',
       'Usage savon et eau', 'Céphalée', 'Douleurs articulaires', 'Fièvre',
       'Douleurs musculaires', 'Douleurs abdominales', 'Toux', 'Rhinorrhée',
       'Anosmie', 'Ageusie', 'Frissons', 'Nausées', 'Dyspnée', 'Vomissements',
       'Sueurs', 'Eruption cutanée', 'Odynophagie', 'Conjonctivite',
       'Rhinorragie', 'Serologie'],
      dtype='object')

In [8]:
# -- Definition des groupes de variables ----
COMOR_COLS = ['Diabte', 'HTA', 'Maladie_cardiaque']

SYMP_COLS = [
            'Céphalée','Fièvre','Toux','Anosmie','Ageusie',
            'Douleurs articulaires','Douleurs musculaires','Douleurs abdominales',
            'Rhinorrhée','Frissons','Nausées','Dyspnée','Vomissements','Sueurs',
            'Eruption cutanée','Odynophagie','Conjonctivite','Rhinorragie'
        ]

PREV_COLS = [
    'Port du masque a domicile',   'Port masque exterieur',
    'Usage du Gel hydroalcoolique','Usage savon et eau',
]

# Colonnes reelles du fichier (noms avec accents)
SYMP_REAL  = [c for c in df.columns if c in [
    'Cephalee','Douleurs articulaires','Fievre','Douleurs musculaires',
    'Douleurs abdominales','Toux','Rhinorrhee','Anosmie','Ageusie',
    'Frissons','Nausees','Dyspnee','Vomissements','Sueurs',
    'Eruption cutanee','Odynophagie','Conjonctivite','Rhinorragie',
] or c in [
            'Céphalée','Fièvre','Toux','Anosmie','Ageusie',
            'Douleurs articulaires','Douleurs musculaires','Douleurs abdominales',
            'Rhinorrhée','Frissons','Nausées','Dyspnée','Vomissements','Sueurs',
            'Eruption cutanée','Odynophagie','Conjonctivite','Rhinorragie'
        ]]

# Utiliser les vraies colonnes du dataframe
symptom_cols  = [c for c in df.columns if c in [
            'Céphalée','Fièvre','Toux','Anosmie','Ageusie',
            'Douleurs articulaires','Douleurs musculaires','Douleurs abdominales',
            'Rhinorrhée','Frissons','Nausées','Dyspnée','Vomissements','Sueurs',
            'Eruption cutanée','Odynophagie','Conjonctivite','Rhinorragie'
        ]]
# Fallback avec noms accentues
if not symptom_cols:
    symptom_cols = [
        c for c in df.columns if c in [
            'Céphalée','Fièvre','Toux','Anosmie','Ageusie',
            'Douleurs articulaires','Douleurs musculaires','Douleurs abdominales',
            'Rhinorrhée','Frissons','Nausées','Dyspnée','Vomissements','Sueurs',
            'Eruption cutanée','Odynophagie','Conjonctivite','Rhinorragie'
        ]
    ]
comorbidity_cols = [c for c in df.columns if c in ['Diabte','HTA','Maladie_cardiaque']]
preventive_cols  = [c for c in df.columns if c in [
    'Port du masque à domicile', 'Port masque exterieur',
    'Usage du Gel hydroalcoolique', 'Usage savon et eau'
]]
print(f'Symptomes         : {len(symptom_cols)} colonnes')
print(f'Comorbidites      : {len(comorbidity_cols)} colonnes')
print(f'Comportements     : {len(preventive_cols)} colonnes')


Symptomes         : 18 colonnes
Comorbidites      : 3 colonnes
Comportements     : 4 colonnes


In [9]:
# -- Fonction d'encodage binaire ------
# Les variables cliniques sont au format Oui/Non/Rare/Frequent.
# On les convertit en 0/1 pour les calculs statistiques.

def encode_binary(val):
    """Oui/Rare/Frequent -> 1  |  Non -> 0  |  NaN -> NaN"""
    if pd.isna(val):
        return np.nan
    return 1 if str(val).strip() in ['Oui', 'Rare', 'Fréquent', 'Frequent'] else 0

# Matrices binaires globales
sym_bin  = df[symptom_cols].apply(lambda col: col.map(encode_binary))
com_bin  = df[comorbidity_cols].apply(lambda col: col.map(encode_binary))
prev_bin = df[preventive_cols].apply(lambda col: col.map(encode_binary))

print('Encodage binaire cree pour symptomes, comorbidites et comportements.')


Encodage binaire cree pour symptomes, comorbidites et comportements.


---
## 3. Profil Socio-Demographique

> Analyse de la **structure demographique** : sexe, age, categorie d'age, profession, statut matrimonial.


In [10]:
# -- Plot 3.1 : Sexe & Categorie d'age --
# Deux graphiques complementaires pour une vue d'ensemble rapide

sexe_cnt = df['Sexe'].value_counts()
age_cnt  = df['Categorie_age'].value_counts().reindex(
    ['Enfant','Adolescent','Adulte','Personne âgée'], fill_value=0)

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'pie'},{'type':'bar'}]],
    subplot_titles=['Répartition par sexe', 'Répartition par catégorie d\'age']
)
fig.add_trace(go.Pie(
    labels=sexe_cnt.index, values=sexe_cnt.values, hole=0.4,
    marker_colors=['#3498DB','#E91E63'],
    textinfo='percent+label', showlegend=False
), row=1, col=1)
fig.add_trace(go.Bar(
    x=age_cnt.index, y=age_cnt.values,
    text=[f'{v}<br>({v/n*100:.1f}%)' for v in age_cnt.values],
    textposition='auto',
    marker_color=['#FF9800','#4CAF50','#9C27B0','#00BCD4'],
    showlegend=False
), row=1, col=2)
fig.update_layout(
    title_text='<b>Sexe & Catégorie d\'age</b>',
    height=460, template='plotly_white', showlegend=False
)
fig.update_yaxes(title_text='Effectif', row=1, col=2)
fig.show()


In [11]:
# -- Plot 3.2 : Pyramide des ages --------
# Representation classique en epidemiologie :
# Femmes a droite, Hommes a gauche, tranches de 10 ans

df['Tranche_age'] = pd.cut(df['Age'], bins=range(0, 110, 10),
                            right=False,
                            labels=[f'{i}-{i+9}' for i in range(0, 100, 10)])
pyramid = df.groupby(['Tranche_age','Sexe'], observed=True).size().unstack(fill_value=0)
tranches = pyramid.index.astype(str).tolist()

h_vals = -pyramid.get('Homme', pd.Series([0]*len(pyramid))).values
f_vals  =  pyramid.get('Femme', pd.Series([0]*len(pyramid))).values

fig_pyr = go.Figure([
    go.Bar(y=tranches, x=h_vals, name='Homme', orientation='h',
           marker_color='#3498DB',
           customdata=abs(h_vals),
           hovertemplate='Homme | %{y} ans<br>n=%{customdata}<extra></extra>'),
    go.Bar(y=tranches, x=f_vals,  name='Femme', orientation='h',
           marker_color='#E91E63',
           hovertemplate='Femme | %{y} ans<br>n=%{y}<extra></extra>')
])

max_v = max(abs(h_vals).max(), f_vals.max())
ticks = list(range(-int(max_v), int(max_v)+1, 100))

fig_pyr.update_layout(
    title='<b>Pyramide des Ages</b>',
    barmode='overlay', bargap=0.1, height=500, template='plotly_white',
    xaxis=dict(title='Effectif', tickvals=ticks, ticktext=[str(abs(v)) for v in ticks],
               zeroline=True, zerolinewidth=2, zerolinecolor='black'),
    yaxis_title='Tranche d\'age'
)
fig_pyr.show()


In [12]:

fig_age = px.histogram(df, x='Age', nbins=30, marginal='box',
                       title='Distribution de l\'âge',
                       color_discrete_sequence=["#3dadc6"])
fig_age.add_vline(x=df['Age'].mean(), line_dash="dash", line_color="red",
                  annotation_text=f"Moyenne = {df['Age'].mean():.1f}")
fig_age.add_vline(x=df['Age'].median(), line_dash="dash", line_color="blue",
                  annotation_text=f"Médiane = {df['Age'].median():.1f}")
fig_age.show()


In [13]:
# -- Plot 3.3 : Distribution de l'age par sexe --
# Histogramme avec boxplot marginal = vision complete de la distribution

fig_age = px.histogram(
    df, x='Age', nbins=30, marginal='box',
    color='Sexe',
    color_discrete_map={'Homme':'#3498DB','Femme':'#E91E63'},
    barmode='overlay', opacity=0.72,
    title='<b>Distribution de l\'age par sexe</b>',
    labels={'Age': 'Age (annees)', 'count': 'Effectif'},
    template='plotly_white', height=440
)
fig_age.add_vline(x=df['Age'].mean(), line_dash='dash', line_color='#2C3E50',
                  annotation_text=f'Moy={df["Age"].mean():.1f}')
fig_age.add_vline(x=df['Age'].median(), line_dash='dot', line_color='#E67E22',
                  annotation_text=f'Med={df["Age"].median():.0f}')
fig_age.show()
print(f'Age -- Moy: {df["Age"].mean():.1f} | Med: {df["Age"].median():.0f} | Min-Max: {int(df["Age"].min())}-{int(df["Age"].max())} ans')


Age -- Moy: 26.6 | Med: 18 | Min-Max: 1-100 ans


In [14]:
# -- Plot 3.4 : Profession & Statut matrimonial --
total = len(df)

prof_cnt = df['Profession'].value_counts().reset_index()
prof_cnt.columns = ['Profession','Effectif']
prof_cnt['Pct'] = (prof_cnt['Effectif']/total*100).round(1)
prof_cnt['label'] = prof_cnt['Effectif'].astype(str) + ' (' + prof_cnt['Pct'].astype(str) + '%)'

fig_prof = px.bar(
    prof_cnt, x='Effectif', y='Profession', orientation='h',
    text='label', title='<b>Repartition par profession</b>',
    color='Effectif', color_continuous_scale='Blues',
    template='plotly_white', height=440
)
fig_prof.update_traces(textposition='auto', textangle=0)
fig_prof.update_layout(yaxis={'categoryorder':'total ascending'}, coloraxis_showscale=False)
fig_prof.show()

# Statut matrimonial
statut_cnt = df['Statut_matrimonial'].value_counts().reset_index()
statut_cnt.columns = ['Statut','Effectif']
statut_cnt['Pct'] = (statut_cnt['Effectif']/total*100).round(1)
fig_statut = px.pie(
    statut_cnt, names='Statut', values='Effectif',
    hole=0.4, title='<b>Repartition par statut matrimonial</b>',
    color_discrete_sequence=px.colors.qualitative.Set3,
    template='plotly_white'
)
fig_statut.update_traces(textposition='inside', texttemplate='%{label}<br>%{percent:.1%}')
fig_statut.show()


In [15]:

# Ordre souhaité
order_agecat = ['Enfant', 'Adolescent', 'Adulte', 'Personne âgée']

# Compter les effectifs et pourcentages (réindexation pour respecter l'ordre)
counts = df['Categorie_age'].value_counts().reindex(order_agecat, fill_value=0)
percent = df['Categorie_age'].value_counts(normalize=True).reindex(order_agecat, fill_value=0) * 100

# Construire le DataFrame pour Plotly
agecat_counts = counts.reset_index()
agecat_counts.columns = ['Categorie_age', 'Effectif']
agecat_counts['Pourcentage'] = percent.values  # valeurs dans le même ordre

# Créer l'étiquette combinée : "Effectif (X.X%)"
agecat_counts['label'] = (
    agecat_counts['Effectif'].astype(str) 
    + ' (' 
    + agecat_counts['Pourcentage'].round(1).astype(str) 
    + '%)'
)

# Tracer le graphique
fig_agecat = px.bar(
    agecat_counts,
    x='Categorie_age',
    y='Effectif',
    text='label',
    title="Répartition par catégorie d'âge",
    color='Categorie_age',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# Positionner le texte à l'intérieur des barres
fig_agecat.update_traces(textposition='auto',textangle=0)

# Ajuster la mise en page
fig_agecat.update_layout(
    template='plotly_white',
    yaxis_title='Effectif'
)

# Afficher
fig_agecat.show()

In [16]:
# -- Plot 3.5 : Sunburst Sexe -> Categorie age -> Statut --
# Le sunburst permet de visualiser 3 niveaux hierarchiques simultanement.
# Lecture : chaque secteur represente une sous-population.

fig_sun = px.sunburst(
    df, path=['Sexe','Categorie_age','Statut_matrimonial'],
    title='<b>Sunburst : Sexe > Cat. d\'age > Statut matrimonial</b>',
    color='Sexe',
    color_discrete_map={'Homme':'#3498DB','Femme':'#E91E63'},
    template='plotly_white', height=580
)
fig_sun.update_traces(
    textinfo='label+percent parent',
    hovertemplate='<b>%{label}</b><br>n=%{value}<br>%{percentParent:.1%} du parent<extra></extra>'
)
fig_sun.show()


---
## 4. Distribution Geographique

> Localisation des individus, taux de positivite par quartier et relation population/cas.


In [17]:
# -- Statistiques par quartier --------
quartier_stats = df.groupby('Quartier_corrige').agg(
    Lat_mean=('Latitude', 'mean'),
    Lon_mean=('Longitude', 'mean'),
    Total=('Serologie', 'count'),
    Positifs=('Serologie', lambda x: (x == 'Positif').sum()),
    Population=('Population', 'first')
).reset_index()
quartier_stats['Taux_positivite'] = (
    quartier_stats['Positifs'] / quartier_stats['Total'] * 100).round(1)

print(f'Nombre de quartiers : {len(quartier_stats)}')
print('Top 10 par taux de positivite :')
display(quartier_stats.nlargest(10,'Taux_positivite')
        [['Quartier_corrige','Total','Positifs','Taux_positivite']]
        .reset_index(drop=True))


Nombre de quartiers : 30
Top 10 par taux de positivite :


,Quartier_corrige,Total,Positifs,Taux_positivite
0,GAMKALLEY GOLLEY,165,117,70.9
1,SAGUIA,117,76,65.0
2,Nord faisceau,120,68,56.7
3,Boukoki IV,140,78,55.7
4,Dar es salam,96,52,54.2
5,Banga bana,126,66,52.4
6,Lamorde,152,79,52.0
7,Pays bas,183,95,51.9
8,Kirkissoye,228,115,50.4
9,Boukoki II,57,26,45.6


In [18]:
# -- Plot 4.1 : Carte interactive des individus --
# Echantillon de 500 points pour des performances optimales

sample_map = df.sample(min(1000, len(df)), random_state=42)
fig_map1 = px.scatter_mapbox(
    sample_map, lat='Latitude', lon='Longitude',
    color='Serologie',
    color_discrete_map={'Positif':'#E74C3C','Négatif':'#2ECC71'},
    zoom=11, mapbox_style='carto-positron',
    title='<b>Localisation des individus</b> (echantillon de 1000)',
    hover_data=['Quartier_corrige','Sexe','Age','Serologie'],
    opacity=0.7
)
fig_map1.update_layout(margin=dict(l=0,r=0,t=50,b=0), height=520)
fig_map1.show()


In [19]:
# -- Plot 4.2 : Carte des quartiers (taille = effectif, couleur = taux) --
# Chaque point = un quartier. Taille = nbre teste. Couleur = taux de positivite.

fig_map2 = px.scatter_mapbox(
    quartier_stats,
    lat='Lat_mean', lon='Lon_mean',
    size='Total', color='Taux_positivite',
    hover_name='Quartier_corrige',
    hover_data={'Total':True,'Positifs':True,'Taux_positivite':True},
    color_continuous_scale='RdYlGn_r',
    mapbox_style='carto-positron', zoom=11,
    title='<b>Taux de positivite par quartier</b> (Taille = nbre teste)',
    size_max=40
)
fig_map2.update_layout(margin=dict(l=0,r=0,t=60,b=0), height=520)
fig_map2.show()


In [20]:
# -- Plot 4.3 : Barres horizontales du taux de positivite par quartier --
fig_taux = px.bar(
    quartier_stats.sort_values('Taux_positivite', ascending=True),
    x='Taux_positivite', y='Quartier_corrige', orientation='h',
    text='Taux_positivite',
    title='<b>Taux de positivite par quartier</b>',
    color='Taux_positivite', color_continuous_scale='RdYlGn_r',
    labels={'Taux_positivite':'Taux (%)','Quartier_corrige':''},
    template='plotly_white', height=600
)
fig_taux.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_taux.update_layout(coloraxis_showscale=False)
fig_taux.show()


In [21]:
# -- Plot 4.4 : Population vs Cas positifs + regression lineaire --
# Ce graphique montre si les quartiers plus peuples ont plus de cas.

from sklearn.linear_model import LinearRegression
X_reg = quartier_stats['Population'].values.reshape(-1,1)
y_reg = quartier_stats['Positifs'].values
model = LinearRegression().fit(X_reg, y_reg)
x_line = np.linspace(0, quartier_stats['Population'].max(), 100)
y_line = model.predict(x_line.reshape(-1,1))
r2 = model.score(X_reg, y_reg)

fig_scat = px.scatter(
    quartier_stats, x='Population', y='Positifs',
    size='Total', color='Taux_positivite',
    hover_name='Quartier_corrige',
    color_continuous_scale='Reds',
    title=f'<b>Population vs Cas positifs</b> (R2={r2:.3f})',
    template='plotly_white', height=460
)
fig_scat.add_trace(go.Scatter(
    x=x_line, y=y_line, mode='lines',
    name=f'Regression (R2={r2:.3f})',
    line=dict(color='#2C3E50', dash='dash', width=2)
))
fig_scat.show()
print(f'Correlation Population-Cas positifs : R2 = {r2:.3f}')


Correlation Population-Cas positifs : R2 = 0.086


In [22]:
top15_taux = quartier_stats.sort_values('Taux_positivite', ascending=False).head(15)
fig_taux = px.bar(top15_taux, x='Taux_positivite', y='Quartier_corrige',
                  orientation='h', text='Taux_positivite',
                  title='Taux de positivité par quartier (top 15)',
                  color='Taux_positivite', color_continuous_scale='Reds')
fig_taux.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_taux.update_layout(xaxis_title='Taux (%)', yaxis_title='')
fig_taux.show()

---
## 5. Distribution Serologique

> **Variable cible** : `Serologie` (Positif / Negatif)  
> Distribution globale et associations avec les principales variables demographiques.


In [23]:
# -- Statistiques globales ------------
sero_cnt = df['Serologie'].value_counts()
n_pos    = sero_cnt.get('Positif', 0)
n_neg    = sero_cnt.get('Négatif', 0)
prev     = n_pos / n * 100

print('=' * 55)
print(f'  Total individus testes  : {n}')
print(f'  Cas positifs            : {n_pos}  ({prev:.1f}%)')
print(f'  Cas negatifs            : {n_neg}  ({n_neg/n*100:.1f}%)')
print('=' * 55)


  Total individus testes  : 4318
  Cas positifs            : 1470  (34.0%)
  Cas negatifs            : 2848  (66.0%)


In [24]:


# --- Graphique 1 : répartition par sexe ---

sexe_counts = df['Sexe'].value_counts().reset_index()
sexe_counts.columns = ['Sexe', 'Effectif']
sexe_counts['Pourcentage'] = (sexe_counts['Effectif'] / total * 100).round(1)
sexe_counts['label'] = sexe_counts['Effectif'].astype(str) + ' (' + sexe_counts['Pourcentage'].astype(str) + '%)'

# --- Graphique 2 : sérologie par sexe ---
sero_sex = df.groupby(['Serologie', 'Sexe']).size().reset_index(name='Effectif')
# Calcul du pourcentage par sexe (total par sexe)
total_par_sexe = df.groupby('Sexe').size().reset_index(name='Total_sexe')
sero_sex = sero_sex.merge(total_par_sexe, on='Sexe')
sero_sex['Pourcentage'] = (sero_sex['Effectif'] / sero_sex['Total_sexe'] * 100).round(1)
sero_sex['label'] = sero_sex['Effectif'].astype(str) + ' (' + sero_sex['Pourcentage'].astype(str) + '%)'

# Création des subplots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Répartition par sexe", "Résultats sérologiques par sexe")
)

# Graphique 1
fig.add_trace(
    go.Bar(
        x=sexe_counts['Sexe'],
        y=sexe_counts['Effectif'],
        text=sexe_counts['label'],
        textposition='auto',
        marker_color=['#3498db', '#e91e63'],
        name='Sexe',
        showlegend=False
    ),
    row=1, col=1
)

# Graphique 2 : une trace par sérologie
for sero in sero_sex['Serologie'].unique():
    data = sero_sex[sero_sex['Serologie'] == sero]
    color = '#2ecc71' if sero == 'Positif' else '#e74c3c'
    fig.add_trace(
        go.Bar(
            x=data['Sexe'],
            y=data['Effectif'],
            name=sero,
            text=data['label'],
            textposition='auto',
            marker_color=color
        ),
        row=1, col=2
    )

# Mise en page
fig.update_layout(
    title="Analyse du sexe et des résultats sérologiques",
    template="plotly_white",
    height=500,
    barmode='group'  # pour le second graphique (groupé)
)

# Ajuster les axes
fig.update_xaxes(title_text="Sexe", row=1, col=1)
fig.update_yaxes(title_text="Effectif", row=1, col=1)
fig.update_xaxes(title_text="Sexe", row=1, col=2)
fig.update_yaxes(title_text="Effectif", row=1, col=2)

fig.show()

In [25]:
# -- Plot 5.3 : Violin plot - age selon la serologie --
# Compare la distribution d'age entre positifs et negatifs
# Test statistique : Mann-Whitney (non-parametrique)

from scipy.stats import mannwhitneyu
ages_pos = df[df['Serologie']=='Positif']['Age'].dropna()
ages_neg = df[df['Serologie']=='Négatif']['Age'].dropna()
_, p_mw  = mannwhitneyu(ages_pos, ages_neg, alternative='two-sided')

fig_vio = px.violin(
    df, x='Serologie', y='Age', color='Serologie',
    box=True, points='outliers',
    color_discrete_map={'Positif':'#E74C3C','Négatif':'#2ECC71'},
    title=f'<b>Distribution de l\'age selon la serologie</b><br>',
    template='plotly_white', height=470
)
fig_vio.update_traces(meanline_visible=True)
fig_vio.show()
print(f'Age moyen -- Positifs: {ages_pos.mean():.1f} ans | Negatifs: {ages_neg.mean():.1f} ans')


Age moyen -- Positifs: 30.6 ans | Negatifs: 24.5 ans


In [26]:
# -- Plot 5.4 : Taux de positivite par profession et statut --

def taux_pos(df, var):
    ct = df.groupby(var)['Serologie'].apply(
        lambda x: round((x=='Positif').sum()/len(x)*100, 1)
    ).reset_index()
    ct.columns = [var,'Taux']
    return ct.sort_values('Taux', ascending=True)

fig_tp = make_subplots(1, 2,
    subplot_titles=['Taux par profession','Taux par statut matrimonial'])

for idx, (var, pos) in enumerate([('Profession',(1,1)),('Statut_matrimonial',(1,2))]):
    tp = taux_pos(df, var)
    fig_tp.add_trace(go.Bar(
        y=tp[var], x=tp['Taux'], orientation='h',
        marker=dict(color=tp['Taux'], colorscale='RdYlGn_r'),
        text=[f'{v}%' for v in tp['Taux']],
        textposition='auto', showlegend=False
    ), row=pos[0], col=pos[1])

fig_tp.update_layout(
    title='<b>Taux de positivite par sous-groupes</b>',
    template='plotly_white', height=420
)
fig_tp.update_xaxes(title_text='Taux (%)')
fig_tp.show()


---
## 6. Analyse des Symptomes

> Frequence, co-occurrence et association des **18 symptomes** cliniques avec la serologie.


In [27]:
# -- Frequence globale des symptomes ----
sym_freq = sym_bin.sum().sort_values(ascending=False).reset_index()
sym_freq.columns = ['Symptome','Effectif']
sym_freq['Pourcentage'] = (sym_freq['Effectif']/n*100).round(1)

print('Prevalence des symptomes dans la population totale :')
display(sym_freq.set_index('Symptome'))


Prevalence des symptomes dans la population totale :


,Effectif,Pourcentage
Symptome,,
Céphalée,822,19.0
Douleurs articulaires,551,12.8
Fièvre,498,11.5
Douleurs musculaires,460,10.7
Douleurs abdominales,388,9.0
Rhinorrhée,285,6.6
Toux,203,4.7
Odynophagie,88,2.0
Eruption cutanée,82,1.9


In [28]:
# -- Plot 6.1 : Frequence des symptomes -- barres colorees --
fig_sym = px.bar(
    sym_freq, x='Effectif', y='Symptome', orientation='h',
    text=sym_freq.apply(lambda r: f"{r['Effectif']} ({r['Pourcentage']}%)", axis=1),
    title='<b>Frequence des Symptomes dans la Population</b>',
    color='Effectif', color_continuous_scale='Plasma',
    template='plotly_white', height=600,
    labels={'Effectif':"Nombre d'individus",'Symptome':''}
)
fig_sym.update_traces(textposition='auto')
fig_sym.update_layout(yaxis={'categoryorder':'total ascending'},
                      coloraxis_showscale=False, margin=dict(r=180))
fig_sym.show()


In [29]:
# sym_freq est déjà défini dans la section 6
# Il contient : 'Symptome', 'Effectif', 'Pourcentage'

# Création de la colonne texte "Effectif (X.X%)"
sym_freq['texte'] = sym_freq['Effectif'].astype(str) + ' (' + sym_freq['Pourcentage'].astype(str) + '%)'

fig_symp_freq = px.bar(
    sym_freq,
    x='Effectif',
    y='Symptome',               
    text='texte',
    orientation='h',
    title='Fréquence des symptômes (%)',
    color='Effectif',
    color_continuous_scale='Viridis',
    labels={'Effectif': 'Nombre de personnes', 'Pourcentage': 'Pourcentage (%)'}
)

fig_symp_freq.update_traces(textposition='auto')
fig_symp_freq.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    xaxis_title='Nombre de personnes'
)
fig_symp_freq.show()

In [30]:
# -- Plot 6.2 : Treemap des symptomes --
# Taille des rectangles proportionnelle a la frequence
fig_tree = px.treemap(
    sym_freq, path=['Symptome'], values='Effectif',
    color='Effectif', color_continuous_scale='Blues',
    title='<b>Treemap -- Proportion relative des symptomes</b>',
    hover_data={'Pourcentage':True}
)
fig_tree.update_traces(
    textinfo='label+percent entry',
    hovertemplate='<b>%{label}</b><br>n=%{value}<br>%{percentEntry:.1%}<extra></extra>'
)
fig_tree.update_layout(margin=dict(t=50,l=25,r=25,b=25), height=550)
fig_tree.show()


In [31]:
# -- Plot 6.3 : Differentiel de positivite par symptome --
# Pour chaque symptome : taux de positivite (avec) - taux de positivite (sans)
# Barres rouges = symptome associe a la positivite

diff_list = []
for sym in sym_bin.columns:
    avec  = df[sym_bin[sym]==1]['Serologie']
    sans  = df[sym_bin[sym]==0]['Serologie']
    t_avec = (avec=='Positif').sum()/len(avec)*100 if len(avec)>0 else 0
    t_sans = (sans=='Positif').sum()/len(sans)*100 if len(sans)>0 else 0
    diff_list.append({'Symptome':sym, 'Avec':round(t_avec,1),
                      'Sans':round(t_sans,1), 'Diff':round(t_avec-t_sans,1)})

df_diff = pd.DataFrame(diff_list).sort_values('Diff')

fig_dif = px.bar(
    df_diff, x='Diff', y='Symptome', orientation='h',
    color='Diff', color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    title='<b>Exces de positivite lie a chaque symptome</b><br>'
          '<sup>= Taux(avec symptome) - Taux(sans symptome)</sup>',
    text=df_diff['Diff'].apply(lambda x: f'+{x}%' if x>0 else f'{x}%'),
    template='plotly_white', height=560
)
fig_dif.update_traces(textposition='outside')
fig_dif.add_vline(x=0, line_color='black', line_width=1.5)
fig_dif.update_layout(coloraxis_showscale=False)
fig_dif.show()


In [32]:
# -- Plot 6.4 : Heatmap de correlation entre symptomes --
# Corrélation de Pearson : +1 = toujours ensemble, 0 = independants

corr_sym = sym_bin.corr()
fig_corr = px.imshow(
    corr_sym, text_auto='.2f', aspect='auto',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='<b>Matrice de Correlation des Symptomes</b>'
)
fig_corr.update_layout(height=700, width=850)
fig_corr.show()


In [33]:
# -- Plot 6.5 : Co-occurrence -- Top 15 paires de symptomes --
cooc = {}
cols_list = sym_bin.columns.tolist()
for i, s1 in enumerate(cols_list):
    for s2 in cols_list[i+1:]:
        cnt = int(((sym_bin[s1]==1) & (sym_bin[s2]==1)).sum())
        if cnt > 0:
            cooc[(s1,s2)] = cnt

top15 = sorted(cooc.items(), key=lambda x: x[1], reverse=True)[:15]
df_cooc = pd.DataFrame({
    'Paire': [f'{s1}  /  {s2}' for (s1,s2),_ in top15],
    'Effectif': [cnt for _,cnt in top15]
})

fig_cooc = px.bar(
    df_cooc, x='Effectif', y='Paire', orientation='h',
    text='Effectif', title='<b>Top 15 Paires de Symptomes Co-occurrents</b>',
    color='Effectif', color_continuous_scale='Viridis',
    template='plotly_white', height=550
)
fig_cooc.update_traces(textposition='auto')
fig_cooc.update_layout(yaxis={'categoryorder':'total ascending'},
                       coloraxis_showscale=False)
fig_cooc.show()


In [34]:
# -- Plot 6.6 : Histogramme du nombre de symptomes par individu --
n_sym_person = sym_bin.sum(axis=1)
fig_nsym = px.histogram(
    x=n_sym_person, nbins=19, marginal='box',
    color_discrete_sequence=['#9B59B6'],
    title='<b>Nombre de Symptomes par Individu</b>',
    labels={'x':'Nombre de symptomes','count':'Effectif'},
    template='plotly_white', height=400
)
fig_nsym.add_vline(x=n_sym_person.mean(), line_dash='dash', line_color='red',
                   annotation_text=f'Moy={n_sym_person.mean():.2f}')
fig_nsym.show()
print(f'Nombre moyen de symptomes par individu : {n_sym_person.mean():.2f}')
print(f'Individus sans aucun symptome : {(n_sym_person==0).sum()} ({(n_sym_person==0).mean()*100:.1f}%)')


Nombre moyen de symptomes par individu : 0.86
Individus sans aucun symptome : 2432 (56.3%)


---
## 7. Analyse des Comorbidites

> Les comorbidites (Diabete, HTA, Maladie cardiaque) sont des facteurs de risque de forme grave.
> On analyse leur prevalence et leur association avec la serologie.


In [35]:
# -- Frequence des comorbidites --------
com_freq = com_bin.sum().reset_index()
com_freq.columns = ['Comorbidite','Effectif']
com_freq['Pct'] = (com_freq['Effectif']/n*100).round(1)
com_labels = {'Diabte':'Diabete','HTA':'HTA','Maladie_cardiaque':'Cardiopathie'}
com_freq['Nom'] = com_freq['Comorbidite'].map(com_labels)
print('Prevalence des comorbidites :')
display(com_freq[['Nom','Effectif','Pct']].set_index('Nom'))


Prevalence des comorbidites :


,Effectif,Pct
Nom,,
Diabete,75,1.7
HTA,285,6.6
Cardiopathie,76,1.8


# benesba lel pos cases

In [36]:
# -- Plot 7.1 : Frequence et taux de positivite par comorbidite --

fig_cm = make_subplots(1, 2,
    subplot_titles=['Frequence des comorbidites',
                    'Taux de positivite selon comorbidite'])

# Barres frequence
fig_cm.add_trace(go.Bar(
    x=com_freq['Nom'], y=com_freq['Effectif'],
    text=[f"{e} ({p}%)" for e,p in zip(com_freq['Effectif'],com_freq['Pct'])],
    textposition='auto',
    marker_color=['#E74C3C','#3498DB','#F39C12'],
    showlegend=False
), row=1, col=1)

# Taux de positivite avec / sans
taux_com = []
for col, nom in com_labels.items():
    avec = df[com_bin[col]==1]['Serologie']
    sans = df[com_bin[col]==0]['Serologie']
    t_avec = (avec=='Positif').sum()/len(avec)*100 if len(avec)>0 else 0
    t_sans = (sans=='Positif').sum()/len(sans)*100 if len(sans)>0 else 0
    taux_com.append({'Comorbidite':nom,
                     'Avec':round(t_avec,1),
                     'Sans':round(t_sans,1)})
df_tc = pd.DataFrame(taux_com)

for grp, color in [('Avec','#E74C3C'),('Sans','#2ECC71')]:
    fig_cm.add_trace(go.Bar(
        x=df_tc['Comorbidite'], y=df_tc[grp],
        name=grp, marker_color=color,
        text=[f'{v}%' for v in df_tc[grp]], textposition='auto'
    ), row=1, col=2)

fig_cm.update_layout(
    title='<b>Analyse des Comorbidites</b>',
    barmode='group', template='plotly_white', height=450
)
fig_cm.update_yaxes(title_text='Effectif', row=1, col=1)
fig_cm.update_yaxes(title_text='Taux de positivite (%)', row=1, col=2)
fig_cm.show()


In [37]:
# -- Plot 7.2 : Combinaisons de comorbidites --
n_comor = com_bin.sum(axis=1)

fig_combi = px.histogram(
    x=n_comor, nbins=4,
    color_discrete_sequence=['#E74C3C'],
    title='<b>Nombre de Comorbidites par Individu</b>',
    labels={'x':'Nombre de comorbidites','count':'Effectif'},
    template='plotly_white', height=370
)
fig_combi.update_xaxes(
    tickvals=[0,1,2,3],
    ticktext=['0 -- Aucune','1 -- Une','2 -- Deux','3 -- Trois']
)
fig_combi.show()

for i in range(4):
    c = (n_comor==i).sum()
    print(f'{i} comorbidite(s) : {c} individus ({c/n*100:.1f}%)')


0 comorbidite(s) : 3990 individus (92.4%)
1 comorbidite(s) : 231 individus (5.3%)
2 comorbidite(s) : 86 individus (2.0%)
3 comorbidite(s) : 11 individus (0.3%)


---
## 8. Comportements Preventifs

> Adherence aux mesures barrieres (masque, gel, savon) et association avec la serologie.


In [38]:
# -- Frequence des modalites --------
prev_list = []
for col in preventive_cols:
    cnts = df[col].value_counts().reset_index()
    cnts.columns = ['Modalite','Effectif']
    cnts['Comportement'] = col
    cnts['Pct'] = (cnts['Effectif']/n*100).round(1)
    prev_list.append(cnts)
prev_df = pd.concat(prev_list, ignore_index=True)

fig_prev = px.bar(
    prev_df, x='Comportement', y='Pct',
    color='Modalite', barmode='stack', text='Pct',
    title='<b>Comportements Preventifs -- Repartition des Modalites (%)</b>',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white', height=460
)
fig_prev.update_traces(texttemplate='%{text:.0f}%', textposition='inside')
fig_prev.update_layout(yaxis_range=[0,105], legend_title_text='Modalite',
                       xaxis_title='')
fig_prev.show()


In [39]:
# -- Plot 8.2 : Comportements x Serologie (4 sous-graphiques) --
# Pour chaque comportement : comparaison du taux de positivite par modalite

fig_p2 = make_subplots(2, 2,
    subplot_titles=preventive_cols)
positions = [(1,1),(1,2),(2,1),(2,2)]

for col, pos in zip(preventive_cols, positions):
    ct     = pd.crosstab(df[col], df['Serologie'])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    for sero, color in [('Positif','#E74C3C'),('Négatif','#2ECC71')]:
        if sero in ct_pct.columns:
            fig_p2.add_trace(go.Bar(
                x=ct_pct.index.tolist(), y=ct_pct[sero].values,
                name=sero, marker_color=color,
                text=[f'{v:.1f}%' for v in ct_pct[sero].values],
                textposition='auto',
                showlegend=(pos==(1,1)),
                legendgroup=sero
            ), row=pos[0], col=pos[1])

fig_p2.update_layout(
    title='<b>Taux de positivite selon les comportements preventifs</b>',
    barmode='stack', template='plotly_white', height=580
)
fig_p2.show()


---
## 9. Analyses Croisees — Serologie x Facteurs

> Vue synthetique des associations entre la serologie et les variables socio-demographiques,
> a l'aide d'une fonction generique replicable.


In [40]:
# -- Fonction generique serologie x variable --
def plot_serologie_by_variable(df, variable, title=None, height=500):
    """
    Cree un graphique a barres comparant :
    - Panel gauche  : effectif total vs positifs (barres overlay)
    - Panel droit   : taux de positivite par modalite

    Parametres :
    ------------
    df       : DataFrame source
    variable : nom de la colonne categorielle
    title    : titre du graphique
    height   : hauteur en pixels
    """
    stats = df.groupby(variable).agg(
        Total=('Serologie','count'),
        Positifs=('Serologie', lambda x: (x=='Positif').sum())
    ).reset_index()
    stats['Taux']       = (stats['Positifs']/stats['Total']*100).round(1)
    stats['Pct_global'] = (stats['Total']/len(df)*100).round(1)
    stats = stats.sort_values('Positifs', ascending=False)

    fig = make_subplots(1, 2,
        subplot_titles=[f'Effectifs par {variable}',
                        f'Taux de positivite par {variable}'])

    fig.add_trace(go.Bar(
        y=stats[variable], x=stats['Total'], orientation='h',
        name='Total', marker_color='#BDC3C7',
        text=[f'{t} ({p}%)' for t,p in zip(stats['Total'],stats['Pct_global'])],
        textposition='auto'
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        y=stats[variable], x=stats['Positifs'], orientation='h',
        name='Positifs', marker_color='#E74C3C',
        text=[f'{t} ({p}%)' for t,p in zip(stats['Positifs'],stats['Taux'])],
        textposition='auto'
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        y=stats[variable], x=stats['Taux'], orientation='h',
        marker=dict(color=stats['Taux'], colorscale='RdYlGn_r'),
        text=[f'{t}%' for t in stats['Taux']],
        textposition='outside', showlegend=False
    ), row=1, col=2)

    fig.update_layout(
        title_text=title or f'<b>Serologie selon {variable}</b>',
        barmode='overlay', template='plotly_white', height=height
    )
    fig.update_xaxes(title_text='Effectif', row=1, col=1)
    fig.update_xaxes(title_text='Taux de positivite (%)', row=1, col=2)
    fig.show()

print('Fonction plot_serologie_by_variable() definie.')


Fonction plot_serologie_by_variable() definie.


In [41]:
# -- Serologie x Categorie d'age --
plot_serologie_by_variable(df,'Categorie_age',
    title="<b>Serologie selon la Categorie d'age</b>")


In [42]:
# -- Serologie x Profession --
plot_serologie_by_variable(df,'Profession',
    title='<b>Serologie selon la Profession</b>', height=520)


In [43]:
# -- Serologie x Statut matrimonial --
plot_serologie_by_variable(df,'Statut_matrimonial',
    title='<b>Serologie selon le Statut matrimonial</b>')


---
## 10. Profil Detaille des Cas Positifs

> Restriction aux **cas positifs** pour etablir leur profil type :
> demographie, symptomes, comorbidites, comportements et geographie.


In [44]:
# -- Sous-population positive ---------
df_pos     = df[df['Serologie']=='Positif'].copy().reset_index(drop=True)
n_pos      = len(df_pos)
sym_bin_p  = df_pos[symptom_cols].apply(lambda col: col.map(encode_binary))
com_bin_p  = df_pos[comorbidity_cols].apply(lambda col: col.map(encode_binary))

print(f'Cas positifs : n = {n_pos} ({n_pos/n*100:.1f}% du total)')
print(f'Age moyen    : {df_pos["Age"].mean():.1f} ans (med. {df_pos["Age"].median():.0f})')
print(f'Femmes       : {(df_pos["Sexe"]=="Femme").sum()} ({(df_pos["Sexe"]=="Femme").mean()*100:.1f}%)')
print(f'Symptomatiques: {(sym_bin_p.sum(axis=1)>0).sum()} ({(sym_bin_p.sum(axis=1)>0).mean()*100:.1f}%)')


Cas positifs : n = 1470 (34.0% du total)
Age moyen    : 30.6 ans (med. 25)
Femmes       : 908 (61.8%)
Symptomatiques: 655 (44.6%)


In [45]:
# -- Plot 10.1 : Profil demographique des positifs (4 panneaux) --

fig_pos = make_subplots(
    rows=2, cols=2,
    specs=[[{'type':'pie'},{'type':'bar'}],
           [{'type':'bar'},{'type':'bar'}]],
    subplot_titles=['Sexe',"Cat. d'age",'Top professions','Statut matrimonial']
)
sexe_p = df_pos['Sexe'].value_counts()
fig_pos.add_trace(go.Pie(
    labels=sexe_p.index, values=sexe_p.values, hole=0.4,
    marker_colors=['#3498DB','#E91E63'],
    textinfo='percent+label', showlegend=False
), row=1, col=1)

age_p = df_pos['Categorie_age'].value_counts().reindex(
    ['Enfant','Adolescent','Adulte','Personne âgée'], fill_value=0)
fig_pos.add_trace(go.Bar(
    x=age_p.index, y=age_p.values,
    marker_color=['#FF9800','#4CAF50','#9C27B0','#00BCD4'],
    text=age_p.values, textposition='auto', showlegend=False
), row=1, col=2)

prof_p = df_pos['Profession'].value_counts().head(5)
fig_pos.add_trace(go.Bar(
    x=prof_p.values, y=prof_p.index, orientation='h',
    marker_color='#2ECC71', text=prof_p.values, textposition='auto', showlegend=False
), row=2, col=1)

stat_p = df_pos['Statut_matrimonial'].value_counts()
fig_pos.add_trace(go.Bar(
    x=stat_p.index, y=stat_p.values, marker_color='#F39C12',
    text=stat_p.values, textposition='auto', showlegend=False
), row=2, col=2)

fig_pos.update_layout(
    title=f'<b>Profil Demographique des Cas Positifs (n={n_pos})</b>',
    template='plotly_white', height=680
)
fig_pos.show()


In [46]:
# -- Plot 10.2 : Symptomes chez les positifs -- bulle --
# Axe X = prevalence chez les positifs
# Axe Y = taux de positivite global du symptome
# Taille = nombre de porteurs

sym_pos = sym_bin_p.sum().reset_index()
sym_pos.columns = ['Symptome','Effectif']
sym_pos['Pct_pos'] = (sym_pos['Effectif']/n_pos*100).round(1)
sym_pos['Taux_global'] = sym_pos['Symptome'].apply(
    lambda s: round((df[sym_bin[s]==1]['Serologie']=='Positif').sum()/
                    max((sym_bin[s]==1).sum(),1)*100,1)
)

fig_bub = px.scatter(
    sym_pos, x='Pct_pos', y='Taux_global',
    size='Effectif', color='Taux_global',
    color_continuous_scale='RdYlGn_r',
    hover_name='Symptome',
    text='Symptome',
    title='<b>Symptomes chez les Positifs : Frequence vs Taux global</b>',
    labels={'Pct_pos':'Prevalence chez les positifs (%)','Taux_global':'Taux de positivite global (%)'},
    template='plotly_white', height=540
)
fig_bub.update_traces(textposition='top center', textfont_size=8)
fig_bub.update_layout(coloraxis_showscale=False)
fig_bub.show()


In [47]:
# -- Plot 10.3 : Carte des cas positifs --
sample_pos = df_pos.sample(min(600,n_pos), random_state=42)
fig_mp = px.scatter_mapbox(
    sample_pos, lat='Latitude', lon='Longitude',
    color='Categorie_age',
    color_discrete_sequence=px.colors.qualitative.Bold,
    zoom=11, mapbox_style='carto-positron',
    title=f'<b>Localisation des Cas Positifs</b> (echantillon, colore par age)',
    hover_data=['Quartier_corrige','Age','Sexe'],
    opacity=0.75
)
fig_mp.update_layout(margin=dict(l=0,r=0,t=50,b=0), height=500)
fig_mp.show()


In [48]:
# -- Plot 10.4 : Score symptomatique x Age --
# Est-ce que les individus plus ages presentent plus de symptomes ?

df_pos['N_symptomes'] = sym_bin_p.sum(axis=1)
fig_sc = px.scatter(
    df_pos, x='Age', y='N_symptomes', color='Sexe', opacity=0.45,
    color_discrete_map={'Homme':'#3498DB','Femme':'#E91E63'},
    trendline='ols',
    title='<b>Nb de Symptomes selon l\'Age (cas positifs)</b>',
    labels={'Age':'Age (annees)','N_symptomes':'Nb de symptomes'},
    template='plotly_white', height=450
)
fig_sc.show()

from scipy.stats import pearsonr
r, p = pearsonr(df_pos['Age'].dropna(),
                df_pos.loc[df_pos['Age'].notna(),'N_symptomes'])
print(f'Correlation Age x Nb de symptomes : r={r:.3f} (p={p:.4f})')


Correlation Age x Nb de symptomes : r=0.167 (p=0.0000)


---
## 11. Synthese & Conclusions


In [49]:
# -- Tableau de synthese final --------
sep = '=' * 70
print(sep)
print('   SYNTHESE -- EXPLORATION DES DONNEES COVID-19')
print(sep)
print(f'  Individus testes   : {n}')
print(f'  Cas positifs       : {n_pos}  ({n_pos/n*100:.1f}%)')
print(f'  Cas negatifs       : {n-n_pos}  ({(n-n_pos)/n*100:.1f}%)')
print(f'  Quartiers couverts : {df["Quartier_corrige"].nunique()}')
print()
print(f'  Age moyen          : {df["Age"].mean():.1f} ans (med. {df["Age"].median():.0f})')
print(f'  Part feminine      : {(df["Sexe"]=="Femme").mean()*100:.1f}%')
print()
print('  Top 5 symptomes :')
for _, row in sym_freq.head(5).iterrows():
    print(f'    {row["Symptome"]:<30} : {row["Effectif"]:>5} ({row["Pourcentage"]}%)')
print()
print('  Comorbidites :')
for col, nom in {'Diabte':'Diabete','HTA':'HTA','Maladie_cardiaque':'Cardiopathie'}.items():
    n_c = com_bin[col].sum()
    print(f'    {nom:<20}: {int(n_c):>5} ({n_c/n*100:.1f}%)')
print(sep)
print('  Suite : Notebook 2 -- Analyse par Clusters')
print(sep)


   SYNTHESE -- EXPLORATION DES DONNEES COVID-19
  Individus testes   : 4318
  Cas positifs       : 1470  (34.0%)
  Cas negatifs       : 2848  (66.0%)
  Quartiers couverts : 30

  Age moyen          : 26.6 ans (med. 18)
  Part feminine      : 59.5%

  Top 5 symptomes :
    Céphalée                       :   822 (19.0%)
    Douleurs articulaires          :   551 (12.8%)
    Fièvre                         :   498 (11.5%)
    Douleurs musculaires           :   460 (10.7%)
    Douleurs abdominales           :   388 (9.0%)

  Comorbidites :
    Diabete             :    75 (1.7%)
    HTA                 :   285 (6.6%)
    Cardiopathie        :    76 (1.8%)
  Suite : Notebook 2 -- Analyse par Clusters


---

## Exploration Terminee

> **Resultats principaux :**
> - **4 318 individus** testes, **aucune valeur manquante**
> - **Seroprevalence : 34.0%** — heterogene selon les quartiers
> - **Population jeune** : mediane 18 ans, forte part d'enfants/adolescents
> - **Symptomes principaux** : Cephalee (19%), Douleurs articulaires (15%), Fievre (12%)
> - **Comorbidites rares** mais presents : HTA (6.6%), Diabete (1.7%)
> - **Comportements preventifs variables** : masque exterieur plus adopte que masque domicile

---
*Notebook 1/2 — Suite : `2_Cluster_Analysis_COVID19.ipynb`*


In [50]:
# Source - https://stackoverflow.com/a/52161322
# Posted by TrinTragula
# Retrieved 2026-04-03, License - CC BY-SA 4.0

from nbconvert import HTMLExporter
import codecs
import nbformat

notebook_name = '1_Exploration_des_donnees_FINAL.ipynb'
output_file_name = 'output.html'

exporter = HTMLExporter()
output_notebook = nbformat.read(notebook_name, as_version=4)

output, resources = exporter.from_notebook_node(output_notebook)
codecs.open(output_file_name, 'w', encoding='utf-8').write(output)
